In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import gradio as gr
import re
import matplotlib.pyplot as plt

# ==================== Configuration ====================
class Config:
    SEED = 42
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    MODEL_PATH = './data/f1_model_final_v3.pth'
    DATA_DIR = './data/'
    
    CAT_COLS = ['Grand Prix', 'Team', 'Driver', 'Nationality']
    NUM_COLS = ['year', 'Prev_Year_Driver_PTS', 'Prev_Year_Driver_Pos', 'Driver_Experience_Years',
                'Prev_Year_Team_PTS', 'Prev_Year_Team_Pos', 'Team_Experience_Years',
                'Driver_Prev_Season_FL_Count', 'Is_Home_Race']
    TARGET_COL = 'is_winner'
    
    # Model hyperparameters
    EMB_DIM = 32
    HIDDEN_DIM = 128
    DROPOUT_RATE = 0.4
    BATCH_SIZE = 256
    LEARNING_RATE = 5e-4
    N_EPOCHS = 50
    PATIENCE = 10

# Set random seeds
def set_seeds():
    np.random.seed(Config.SEED)
    torch.manual_seed(Config.SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(Config.SEED)

# ==================== Data Processing ====================
class DataProcessor:
    @staticmethod
    def clean_string(text):
        """Remove extra whitespace"""
        return re.sub(r'\s+', ' ', text).strip() if isinstance(text, str) else text
    
    @staticmethod
    def get_country_from_gp(gp_name):
        """Map GP name to country code"""
        GP_COUNTRY_MAP = {
            'British': 'GBR', 'Great Britain': 'GBR', 'Monaco': 'MON', 'Italian': 'ITA', 'Italy': 'ITA',
            'German': 'GER', 'Germany': 'GER', 'Belgian': 'BEL', 'Belgium': 'BEL', 'French': 'FRA', 'France': 'FRA',
            'Dutch': 'NED', 'Spanish': 'ESP', 'Spain': 'ESP', 'Brazilian': 'BRA', 'Brazil': 'BRA',
            'Japanese': 'JPN', 'Japan': 'JPN', 'Canadian': 'CAN', 'Canada': 'CAN', 'Austrian': 'AUT', 'Austria': 'AUT',
            'Hungarian': 'HUN', 'Hungary': 'HUN', 'Mexican': 'MEX', 'Mexico': 'MEX', 'Australian': 'AUS', 'Australia': 'AUS',
            'United States': 'USA', 'USA': 'USA', 'Swiss': 'SUI', 'Switzerland': 'SUI',
        }
        if not isinstance(gp_name, str):
            return None
        for key, country_code in GP_COUNTRY_MAP.items():
            if key in gp_name:
                return country_code
        return None
    
    @staticmethod
    def fill_missing(df, cat_cols, num_cols):
        """Fill missing values with defaults"""
        for col in cat_cols:
            if col not in df.columns:
                df[col] = 'Unknown'
            df[col] = df[col].fillna('Unknown')
        for col in num_cols:
            if col not in df.columns:
                df[col] = 0
            df[col] = df[col].fillna(0)
        return df

class DataLoaderUtil:
    @staticmethod
    def load_data():
        """Load and clean all CSV data"""
        d = Config.DATA_DIR
        
        # Load data
        winners = pd.read_csv(d + 'winners.csv', encoding='utf-8')
        drivers = pd.read_csv(d + 'drivers_updated.csv', encoding='utf-8')
        teams = pd.read_csv(d + 'teams_updated.csv', encoding='utf-8')
        fastest_laps = pd.read_csv(d + 'fastest_laps_updated.csv', encoding='utf-8')
        
        # Clean text columns
        for df in [winners, drivers, teams, fastest_laps]:
            for col in df.select_dtypes(include='object'):
                df[col] = df[col].map(DataProcessor.clean_string)
        
        # Process years
        winners['year'] = pd.to_datetime(winners['Date'], errors='coerce').dt.year.astype(int)
        drivers.rename(columns={'Car': 'Team'}, inplace=True)
        
        for df in [drivers, teams, fastest_laps]:
            df['year'] = pd.to_numeric(df['year'], errors='coerce').astype(int)
            if 'Pos' in df.columns:
                df['Pos'] = pd.to_numeric(df['Pos'], errors='coerce')
        
        # Exclude Indianapolis 500
        winners = winners[~winners['Grand Prix'].str.contains("Indianapolis 500", na=False)]
        fastest_laps = fastest_laps[~fastest_laps['Grand Prix'].str.contains("Indianapolis 500", na=False)]
        
        return winners, drivers, teams, fastest_laps

# ==================== Feature Engineering ====================
class FeatureEngineer:
    @staticmethod
    def create_lag_features(drivers, teams, fastest_laps, def_pos_drv=50, def_pos_team=20):
        """Create all lag features"""
        # Driver features
        drivers = drivers.sort_values(['Driver', 'year'])
        drivers['Prev_Year_Driver_PTS'] = drivers.groupby('Driver')['PTS'].shift(1).fillna(0)
        drivers['Prev_Year_Driver_Pos'] = drivers.groupby('Driver')['Pos'].shift(1).fillna(def_pos_drv)
        drivers['Driver_Experience_Years'] = drivers['year'] - drivers.groupby('Driver')['year'].transform('min')
        
        # Team features
        teams = teams.sort_values(['Team', 'year'])
        teams['Prev_Year_Team_PTS'] = teams.groupby('Team')['PTS'].shift(1).fillna(0)
        teams['Prev_Year_Team_Pos'] = teams.groupby('Team')['Pos'].shift(1).fillna(def_pos_team)
        teams['Team_Experience_Years'] = teams['year'] - teams.groupby('Team')['year'].transform('min')
        
        # Merge team features
        drivers = drivers.merge(
            teams[['Team', 'year', 'Prev_Year_Team_PTS', 'Prev_Year_Team_Pos', 'Team_Experience_Years']],
            on=['Team', 'year'], how='left'
        ).fillna(0)
        
        # Fastest lap features
        fl = fastest_laps.groupby(['year', 'Driver']).size().reset_index(name='FL_Count')
        fl['Driver_Prev_Season_FL_Count'] = fl.groupby('Driver')['FL_Count'].shift(1).fillna(0)
        drivers = drivers.merge(
            fl[['Driver', 'year', 'Driver_Prev_Season_FL_Count']],
            on=['Driver', 'year'], how='left'
        ).fillna(0)
        
        return drivers
    
    @staticmethod
    def build_model_dataset(winners, drivers):
        """Build complete modeling dataset"""
        data = []
        drivers_by_year = {y: g for y, g in drivers.groupby('year')}
        
        for _, race in winners.iterrows():
            year, gp, winner = race['year'], race['Grand Prix'], race['Winner']
            race_country = DataProcessor.get_country_from_gp(gp)
            
            if year not in drivers_by_year:
                continue
                
            for _, driver_row in drivers_by_year[year].iterrows():
                is_home = 1 if race_country and driver_row['Nationality'] == race_country else 0
                
                data.append({
                    'year': year, 'Grand Prix': gp, 'Driver': driver_row['Driver'], 
                    'Team': driver_row['Team'], 'Nationality': driver_row['Nationality'],
                    'Prev_Year_Driver_PTS': driver_row['Prev_Year_Driver_PTS'],
                    'Prev_Year_Driver_Pos': driver_row['Prev_Year_Driver_Pos'],
                    'Driver_Experience_Years': driver_row['Driver_Experience_Years'],
                    'Prev_Year_Team_PTS': driver_row['Prev_Year_Team_PTS'],
                    'Prev_Year_Team_Pos': driver_row['Prev_Year_Team_Pos'],
                    'Team_Experience_Years': driver_row['Team_Experience_Years'],
                    'Driver_Prev_Season_FL_Count': driver_row['Driver_Prev_Season_FL_Count'],
                    'Is_Home_Race': is_home,
                    'is_winner': int(driver_row['Driver'] == winner)
                })
        
        return pd.DataFrame(data)

# ==================== Preprocessing ====================
class Preprocessor:
    @staticmethod
    def encode_and_scale(train, test, cat_cols, num_cols):
        """Encode categorical and scale numerical features"""
        encoders, cat_dims = {}, {}
        
        # Categorical encoding
        for col in cat_cols:
            le = LabelEncoder()
            train[col] = le.fit_transform(train[col].astype(str))
            test[col] = test[col].map(
                lambda x: le.transform([x])[0] if x in le.classes_ else len(le.classes_)
            )
            encoders[col] = le
            cat_dims[col] = len(le.classes_) + 1
        
        # Numerical scaling
        scaler = StandardScaler()
        train[num_cols] = scaler.fit_transform(train[num_cols])
        test[num_cols] = scaler.transform(test[num_cols])
        
        return train, test, encoders, scaler, cat_dims

# ==================== Dataset and Model ====================
class F1Dataset(Dataset):
    def __init__(self, df, cat_cols, num_cols, target_col):
        self.x_cat = df[cat_cols].values
        self.x_num = df[num_cols].values
        self.y = df[target_col].values
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return (
            torch.tensor(self.x_cat[idx], dtype=torch.long),
            torch.tensor(self.x_num[idx], dtype=torch.float32),
            torch.tensor(self.y[idx], dtype=torch.float32)
        )

class F1DNN(nn.Module):
    def __init__(self, cat_dims, num_num_feats, 
                 emb_dim=Config.EMB_DIM, hidden_dim=Config.HIDDEN_DIM, dropout_rate=Config.DROPOUT_RATE):
        super().__init__()
        self.embeddings = nn.ModuleList([nn.Embedding(dim, emb_dim) for dim in cat_dims])
        self.bn_num = nn.BatchNorm1d(num_num_feats)
        self.fc = nn.Sequential(
            nn.Linear(len(cat_dims) * emb_dim + num_num_feats, hidden_dim),
            nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim // 2, 1)
        )
    
    def forward(self, x_cat, x_num):
        x = torch.cat([emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)], 1)
        x = torch.cat([x, self.bn_num(x_num)], 1)
        return self.fc(x)

# ==================== Training and Evaluation ====================
class ModelTrainer:
    @staticmethod
    def train_model(model, train_loader, test_loader):
        """Train the model with early stopping"""
        optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LEARNING_RATE, weight_decay=1e-4)
        criterion = nn.BCEWithLogitsLoss()
        
        best_test_loss = float('inf')
        no_improve = 0
        train_losses, test_losses = [], []
        
        for epoch in range(Config.N_EPOCHS):
            # Training
            model.train()
            train_loss = 0
            for x_cat, x_num, y_true in train_loader:
                x_cat, x_num, y_true = x_cat.to(Config.DEVICE), x_num.to(Config.DEVICE), y_true.to(Config.DEVICE)
                optimizer.zero_grad()
                loss = criterion(model(x_cat, x_num), y_true.unsqueeze(1))
                loss.backward()
                optimizer.step()
                train_loss += loss.item()
            
            avg_train_loss = train_loss / len(train_loader)
            train_losses.append(avg_train_loss)
            
            # Validation
            model.eval()
            test_loss = 0
            with torch.no_grad():
                for x_cat, x_num, y_true in test_loader:
                    x_cat, x_num, y_true = x_cat.to(Config.DEVICE), x_num.to(Config.DEVICE), y_true.to(Config.DEVICE)
                    loss = criterion(model(x_cat, x_num), y_true.unsqueeze(1))
                    test_loss += loss.item()
            
            avg_test_loss = test_loss / len(test_loader)
            test_losses.append(avg_test_loss)
            
            print(f"Epoch {epoch+1} | Train: {avg_train_loss:.4f} | Test: {avg_test_loss:.4f}")
            
            # Early stopping
            if avg_test_loss < best_test_loss:
                best_test_loss = avg_test_loss
                torch.save(model.state_dict(), Config.MODEL_PATH)
                no_improve = 0
            else:
                no_improve += 1
                if no_improve >= Config.PATIENCE:
                    print("Early stopping.")
                    break
        
        # Save loss curve
        plt.figure()
        plt.plot(train_losses, label='Train')
        plt.plot(test_losses, label='Test')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        plt.grid(True)
        plt.savefig('./data/loss_curve_final.png')
        plt.close()
    
    @staticmethod
    def evaluate_model(model, dataloader, model_path=None):
        """Evaluate model performance"""
        if model_path and os.path.exists(model_path):
            model.load_state_dict(torch.load(model_path, map_location=Config.DEVICE))
        
        model.eval()
        y_true, y_pred = [], []
        
        with torch.no_grad():
            for x_cat, x_num, y in dataloader:
                x_cat, x_num, y = x_cat.to(Config.DEVICE), x_num.to(Config.DEVICE), y.to(Config.DEVICE)
                probs = torch.sigmoid(model(x_cat, x_num))
                pred = (probs > 0.5).squeeze().int()
                y_true.extend(y.cpu().numpy())
                y_pred.extend(pred.cpu().numpy() if pred.ndim > 0 else [pred.item()])
        
        print("Accuracy:", accuracy_score(y_true, y_pred))
        print(classification_report(y_true, y_pred, target_names=['Not Winner', 'Winner'], zero_division=0))
        print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))

# ==================== Gradio Interface ====================
class GradioPredictor:
    def __init__(self):
        self.label_encoders = {}
        self.scaler = None
        self.model = None
        self.driver_info_by_year = {}
        self.all_years = []
        self.all_gps = []
        self.cat_cols_ordered = []
        self.num_cols_ordered = []
    
    def setup(self, label_encoders, scaler, model, drivers_feat, winners):
        """Setup predictor with trained components"""
        self.label_encoders = label_encoders
        self.scaler = scaler
        self.model = model
        self.cat_cols_ordered = Config.CAT_COLS
        self.num_cols_ordered = Config.NUM_COLS
        
        for year, group in drivers_feat.groupby('year'):
            self.driver_info_by_year[year] = group.to_dict('records')
        
        self.all_years = sorted(list(drivers_feat['year'].unique()))
        self.all_gps = sorted(list(winners['Grand Prix'].unique()))
    
    def predict_winner_probabilities(self, year_input, grand_prix_input):
        """Predict winner probabilities for given race"""
        if not self.model or not self.label_encoders or (self.num_cols_ordered and self.scaler is None):
            return "Error: Model or preprocessors not loaded."
        
        try:
            year = int(year_input)
        except:
            return "Error: Invalid year."
        
        if not grand_prix_input:
            return "Error: Grand Prix input empty."
        
        drivers = self.driver_info_by_year.get(year, [])
        if not drivers:
            return f"No driver data for {year}."
        
        self.model.eval()
        results = []
        
        for driver_info in drivers:
            # Categorical features
            cat = [
                self.label_encoders[col].transform([str(driver_info.get(col, 'Unknown'))])[0]
                if str(driver_info.get(col, '')) in set(self.label_encoders[col].classes_)
                else len(self.label_encoders[col].classes_)
                for col in self.cat_cols_ordered
            ]
            
            # Numerical features
            num = [
                float(driver_info.get(col, 0)) if col != 'Is_Home_Race'
                else 1.0 if DataProcessor.get_country_from_gp(grand_prix_input) == driver_info.get('Nationality') else 0.0
                for col in self.num_cols_ordered
            ]
            
            # Transform features
            x_num_df = pd.DataFrame([num], columns=self.num_cols_ordered)
            x_num = torch.tensor(self.scaler.transform(x_num_df), dtype=torch.float32).to(Config.DEVICE)
            x_cat = torch.tensor([cat], dtype=torch.long).to(Config.DEVICE)
            
            # Predict
            with torch.no_grad():
                prob = torch.sigmoid(self.model(x_cat, x_num)).cpu().item()
            
            results.append((driver_info.get('Driver', 'N/A'), driver_info.get('Team', 'N/A'), prob))
        
        # Sort and return top 5
        results.sort(key=lambda x: x[2], reverse=True)
        output = f"Predictions for {grand_prix_input}, {year}:\n"
        for i, (driver, team, prob) in enumerate(results[:5], 1):
            output += f"{i}. {driver} ({team}): {prob:.2%}\n"
        
        return output.strip()
    
    def create_interface(self):
        """Create Gradio interface"""
        with gr.Blocks(theme=gr.themes.Soft()) as demo:
            gr.Markdown("# F1 Grand Prix Winner Predictor (Top 5)")
            
            with gr.Row():
                year_dd = gr.Dropdown(
                    label="Year", 
                    choices=self.all_years, 
                    value=self.all_years[-1] if self.all_years else None
                )
                gp_dd = gr.Dropdown(
                    label="Grand Prix", 
                    choices=self.all_gps, 
                    value=self.all_gps[0] if self.all_gps else None
                )
            
            predict_btn = gr.Button("Predict Probabilities")
            output_tb = gr.Textbox(label="Predicted Top 5", lines=8, interactive=False)
            
            predict_btn.click(
                self.predict_winner_probabilities, 
                inputs=[year_dd, gp_dd], 
                outputs=[output_tb]
            )
            
            with gr.Accordion("Training Information", open=False):
                if os.path.exists("./data/loss_curve_final.png"):
                    gr.Image(value="./data/loss_curve_final.png", label="Loss Curve")
                else:
                    gr.Markdown("Loss curve image not found.")
        
        return demo

# ==================== Main Pipeline ====================
def main():
    """Main execution pipeline"""
    # Setup
    os.makedirs(Config.DATA_DIR, exist_ok=True)
    set_seeds()
    
    # Load and process data
    data_loader_util = DataLoaderUtil()
    winners, drivers, teams, fastest_laps = data_loader_util.load_data()
    
    # Feature engineering
    def_pos_drv = int(drivers['Pos'].max(skipna=True)) + 5 if 'Pos' in drivers else 50
    def_pos_team = int(teams['Pos'].max(skipna=True)) + 5 if 'Pos' in teams else 20
    
    feature_engineer = FeatureEngineer()
    drivers_feat = feature_engineer.create_lag_features(drivers, teams, fastest_laps, def_pos_drv, def_pos_team)
    modeling_df = feature_engineer.build_model_dataset(winners, drivers_feat)
    
    # Train/test split
    modeling_df['race_id'] = modeling_df['year'].astype(str) + "_" + modeling_df['Grand Prix'].astype(str)
    unique_race_ids = modeling_df['race_id'].unique()
    
    if len(unique_race_ids) < 2:
        train_df, test_df = modeling_df.copy(), modeling_df.copy()
    else:
        train_ids, test_ids = train_test_split(unique_race_ids, test_size=0.2, random_state=Config.SEED, shuffle=True)
        train_df = modeling_df[modeling_df['race_id'].isin(train_ids)].copy()
        test_df = modeling_df[modeling_df['race_id'].isin(test_ids)].copy()
    
    # Preprocessing
    train_df = DataProcessor.fill_missing(train_df, Config.CAT_COLS, Config.NUM_COLS)
    test_df = DataProcessor.fill_missing(test_df, Config.CAT_COLS, Config.NUM_COLS)
    
    for df in [train_df, test_df]:
        if 'race_id' in df.columns:
            df.drop(columns=['race_id'], inplace=True)
    
    preprocessor = Preprocessor()
    train_df, test_df, label_encoders, scaler, cat_dims_map = preprocessor.encode_and_scale(
        train_df, test_df, Config.CAT_COLS, Config.NUM_COLS
    )
    
    # Create datasets and loaders
    train_set = F1Dataset(train_df, Config.CAT_COLS, Config.NUM_COLS, Config.TARGET_COL)
    test_set = F1Dataset(test_df, Config.CAT_COLS, Config.NUM_COLS, Config.TARGET_COL)
    
    weights = [1. / (train_df[Config.TARGET_COL].value_counts().get(t, 1) + 1e-6) for t in train_df[Config.TARGET_COL]]
    sampler = WeightedRandomSampler(torch.DoubleTensor(weights), len(weights))
    
    train_loader = DataLoader(train_set, batch_size=Config.BATCH_SIZE, sampler=sampler)
    test_loader = DataLoader(test_set, batch_size=Config.BATCH_SIZE, shuffle=False)
    
    # Model setup
    ordered_cat_dims = [cat_dims_map[c] for c in Config.CAT_COLS]
    num_num_feats = len(Config.NUM_COLS)
    model = F1DNN(ordered_cat_dims, num_num_feats).to(Config.DEVICE)
    
    # Training
    trainer = ModelTrainer()
    train_flag = not os.path.exists(Config.MODEL_PATH)
    
    if train_flag:
        trainer.train_model(model, train_loader, test_loader)
    
    trainer.evaluate_model(model, test_loader, model_path=Config.MODEL_PATH if not train_flag else None)
    
    # Setup Gradio interface
    predictor = GradioPredictor()
    predictor.setup(label_encoders, scaler, model, drivers_feat, winners)
    demo = predictor.create_interface()
    
    # Launch
    demo.launch(share=False)

if __name__ == '__main__':
    main()

c:\Users\lu050\anaconda3\envs\pytorch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


TypeError: DataLoader() takes no arguments